<a href="https://colab.research.google.com/github/werowe/HypatiaAcademy/blob/master/ml/student_random_forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Why Random Forests?

Random Forests are ensemble learning models. They train many decision trees and then combine their predictions.

In this notebook, we use a student example instead of the golf/weather example.

The goal is to predict whether a student will **pass** or **fail** based on simple study-related features:

* **StudyHours** — how many hours the student studied
* **SleepHours** — how many hours the student slept
* **AttendedClass** — whether the student attended class regularly
* **PriorGrade** — the student's previous grade

For classification, each tree votes for a class. The random forest uses the majority vote.

Random forests are useful because a single tree can overfit the training data. But many different trees can reduce variance and make the final prediction more stable.

# Important Note on Synthetic Student Data

Real student performance depends on many factors. In a real project, we would want real historical student data.

Here we create synthetic data only for teaching.

We create features such as study hours, sleep hours, attendance, and prior grade. Then we create a label called **Pass**.

The label is not completely deterministic. A student who studies more and has better attendance is more likely to pass, but there is still some randomness. This makes the data more realistic for a machine learning example.

In [ ]:
import pandas as pd
import numpy as np

# Set a random seed so the results are reproducible.
np.random.seed(42)

# Number of synthetic students.
data_size = 100

# Generate student features.
# StudyHours: number of hours studied before the exam.
study_hours = np.random.randint(0, 9, size=data_size)

# SleepHours: number of hours slept the night before.
sleep_hours = np.random.randint(3, 10, size=data_size)

# AttendedClass: 1 means the student attended class regularly, 0 means not regularly.
attended_class = np.random.choice([0, 1], size=data_size, p=[0.25, 0.75])

# PriorGrade: previous grade from 40 to 100.
prior_grade = np.random.randint(40, 101, size=data_size)

# Create a simple score that influences whether the student passes.
# This is only for generating synthetic labels.
score = (
    study_hours * 8 +
    sleep_hours * 3 +
    attended_class * 15 +
    prior_grade * 0.6
)

# Add random noise so the labels are not perfectly predictable.
noise = np.random.normal(0, 10, size=data_size)

# A student passes if the score plus noise is above a threshold.
pass_exam = ((score + noise) >= 75).astype(int)

# Create the DataFrame.
df = pd.DataFrame({
    'StudyHours': study_hours,
    'SleepHours': sleep_hours,
    'AttendedClass': attended_class,
    'PriorGrade': prior_grade,
    'Pass': pass_exam
})

print(df.head())
print(f"\nValue counts for Pass:")
print(df['Pass'].value_counts())

# Random Forest Principles: Bootstrapping and Feature Sampling

Random forests create many different trees. The trees are different for two main reasons.

## 1. Bootstrapping

Each tree is trained on a random sample of rows from the training data.

This sampling is done **with replacement**. That means the same student row can appear more than once in one tree's training sample, and some student rows may not appear at all.

For example, if the original rows are:

A, B, C, D, E, F

one bootstrap sample might be:

C, A, F, C, E, A

The sample still has six rows, but the order changed, some rows repeated, and some rows disappeared.

## 2. Feature Sampling

At each split, a tree only sees a random subset of the available features.

For example, one tree may split using **StudyHours** and **SleepHours**. Another tree may split using **AttendedClass** and **PriorGrade**.

This makes the trees less correlated. When less correlated trees vote together, the final model is usually more stable than a single decision tree.

In [ ]:
from sklearn.model_selection import train_test_split

# Define features (X) and target label (y).
# X contains the information we use to make the prediction.
X = df[['StudyHours', 'SleepHours', 'AttendedClass', 'PriorGrade']]

# y contains the answer we want to predict.
# 1 means pass. 0 means fail.
y = df['Pass']

# Split the data into training and testing sets.
# The model learns from the training set.
# We evaluate the model on the test set, which it has not seen before.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

# Random Forest Classifier Parameters

We use `RandomForestClassifier` because this is a classification problem.

The target label is **Pass**, which has two possible classes:

* `1` = pass
* `0` = fail

Important parameters:

* `n_estimators`: the number of decision trees in the forest.
* `random_state`: makes the random steps reproducible.
* `bootstrap=True`: each tree uses a bootstrap sample of rows.
* `max_features='sqrt'`: at each split, each tree considers only a random subset of features.

These random choices help create different trees. Different trees make different mistakes, and the forest combines them with majority voting.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Create the random forest classifier.
# n_estimators=100 means the forest will contain 100 decision trees.
rf_classifier = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

print("RandomForestClassifier initialized:")
print(rf_classifier)

In [ ]:
# Train the Random Forest Classifier.
# The model learns patterns between the student features and the Pass label.
rf_classifier.fit(X_train, y_train)

print("RandomForestClassifier trained successfully.")

In [ ]:
# Make predictions on the test set.
# These are predictions for students the model did not see during training.
y_pred = rf_classifier.predict(X_test)

print("Predictions on the test set have been made.")

### Model Evaluation

After training, we evaluate the model on the test set.

The test set contains student rows that were not used during training.

We use `predict()` to generate predictions, and then compare those predictions with the true labels.

For this simple classification example, we calculate accuracy.

Accuracy tells us what percentage of test examples were classified correctly.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(y_test, y_pred)

print(f"Model accuracy on the test set: {accuracy:.2f}")

print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=['Fail', 'Pass']))